# CODY-SAM3 — Master pipeline notebook

End-to-end notebook for the SAM 3 + TabICLv2 pipeline. It is structured into
sections A-F, matching the cody-2 paper's pipeline as closely as possible while
using SAM 3 dense silhouette signals instead of YOLOv8 keypoints:

- **A.** Environment setup + paths
- **B.** Merge SAM 3 timeseries with cody-2 clinician annotations
- **C.** Train per-label TabICLv2 classifiers (Tier 1, 2, 3 in parallel)
- **D.** Inference on the three external datasets (1/2/3)
- **E.** (Optional) Per-site calibration on a small subset
- **F.** Reporting

Run the cells in order. Each section is self-contained and resumable: re-running
B does not re-run A, and so on.

## A. Environment and paths

Edit the paths in cell A.1 to match your local layout. The pipeline expects:

- `SAM3_OUTPUTS_ROOT` — directory containing one subfolder per training
  subject (e.g. `outputs/P1_RPA/...`, `outputs/C1/...`), each subfolder
  containing the `*_sam3_timeseries.xlsx` files produced by your SAM 3 notebook.
- `CODY2_LABELS_ROOT` — directory containing cody-2 labelled files in the
  standard layout: `<root>/dataset_lc/<sid>/*_merged.xlsx`,
  `<root>/dataset_dd/<sid>/*_merged.xlsx`, etc.
- `CODY2_REPO` — path to your cody-pipeline/ folder so we can import
  cody-2's helpers.
- `SAM3_INFER_ROOT_<n>` — directory with SAM 3 outputs for inference dataset
  n (run your SAM 3 notebook on dataset_1/2/3 first).

In [ ]:
from pathlib import Path
import sys, os

# ----- Paths (edit these) -----
SAM3_OUTPUTS_ROOT  = Path(r'C:\Users\<user>\Desktop\sam_3\outputs')
CODY2_LABELS_ROOT  = Path(r'C:\Users\<user>\Desktop\cody_2\labels')
CODY2_REPO         = Path(r'C:\Users\<user>\Desktop\cody_2\cody-pipeline')
MERGED_ROOT        = Path(r'C:\Users\<user>\Desktop\sam_3\merged')
RUNS_ROOT          = Path(r'C:\Users\<user>\Desktop\sam_3\runs')

# Inference-dataset SAM 3 outputs (set when those have been processed)
SAM3_INFER_ROOT_1 = Path(r'C:\Users\<user>\Desktop\sam_3\outputs_dataset_1')
SAM3_INFER_ROOT_2 = Path(r'C:\Users\<user>\Desktop\sam_3\outputs_dataset_2')
SAM3_INFER_ROOT_3 = Path(r'C:\Users\<user>\Desktop\sam_3\outputs_dataset_3')

# Make the pipeline scripts importable
PIPELINE_DIR = Path(__file__).resolve().parent if '__file__' in dir() else Path.cwd().parent
sys.path.insert(0, str(PIPELINE_DIR))

# Quick sanity check
for name, p in [('SAM3 outputs', SAM3_OUTPUTS_ROOT),
                ('cody-2 labels', CODY2_LABELS_ROOT),
                ('cody-2 repo', CODY2_REPO)]:
    print(f'  {name:<18s} {"OK" if p.exists() else "MISSING":<8s} {p}')

## B. Merge SAM 3 timeseries with cody-2 labels

Combines each `<stem>_sam3_timeseries.xlsx` with the matching cody-2
`<stem>_merged.xlsx`. Produces the merged dataset under `MERGED_ROOT/`
in the standard dataset_lc / dataset_dd / dataset_action / dataset_rest /
dataset_posture / dataset_consensus layout. Run once, then this cell can be
skipped on subsequent runs.

Expected runtime: **a few minutes** for the development cohort (~125 videos).

In [ ]:
import subprocess

RUN_MERGE = True   # set to False to skip if MERGED_ROOT is already populated

if RUN_MERGE:
    cmd = [
        sys.executable, str(PIPELINE_DIR / 'merge_sam3_labels.py'),
        '--sam3_root',   str(SAM3_OUTPUTS_ROOT),
        '--labels_root', str(CODY2_LABELS_ROOT),
        '--output_root', str(MERGED_ROOT),
        '--verbose',
    ]
    print('Running:', ' '.join(cmd))
    res = subprocess.run(cmd, capture_output=True, text=True)
    print(res.stdout)
    if res.returncode != 0:
        print('STDERR:', res.stderr)
        raise RuntimeError('merge_sam3_labels failed')

    # Show a one-line summary
    import pandas as pd
    s = pd.read_csv(MERGED_ROOT / '_merge_summary.csv')
    print('\nMerge by dataset_tag and status:')
    print(s.groupby(['tag', 'status']).size().unstack(fill_value=0))

## C. Train per-label TabICLv2 classifiers

Three trainings in parallel: Tier 1 (minimal, 26 signals), Tier 2 (regional, ~66
signals) and Tier 3 (raw, 334 signals). Each one produces a separate `runs/sam3_tierN/`
directory with its own `models/` and `bundle_meta.json`.

Training uses the standard cody-2 protocol: 19 statistical descriptors per signal
per window, negative subsampling with controls preserved, one TabICLv2 per label.

Expected runtime: **15-90 minutes per tier** on an RTX 5080, depending on tier
and number of windows.

In [ ]:
TIERS_TO_TRAIN = [1, 2, 3]    # change to e.g. [2] to train only Tier 2

for tier in TIERS_TO_TRAIN:
    out_dir = RUNS_ROOT / f'sam3_tier{tier}'
    print(f'\n=== Training Tier {tier} -> {out_dir} ===')
    cmd = [
        sys.executable, str(PIPELINE_DIR / 'train_sam3.py'),
        '--train_root', str(MERGED_ROOT),
        '--out_dir',    str(out_dir),
        '--cody2_root', str(CODY2_REPO),
        '--tier',       str(tier),
        '--robust_norm',
        '--use_cache',
    ]
    res = subprocess.run(cmd)
    if res.returncode != 0:
        raise RuntimeError(f'train_sam3 tier {tier} failed')

### C.1. Optional: compare training summary across tiers

Read the per-label `training_label_summary.csv` from each tier run and combine.

In [ ]:
import pandas as pd
rows = []
for tier in TIERS_TO_TRAIN:
    csv = RUNS_ROOT / f'sam3_tier{tier}' / 'training_label_summary.csv'
    if not csv.exists():
        continue
    df = pd.read_csv(csv)
    df['tier'] = tier
    rows.append(df)
if rows:
    all_sum = pd.concat(rows, ignore_index=True)
    print(all_sum.pivot_table(index=['label','level'], columns='tier',
                              values='n_total', aggfunc='first'))

## D. Inference on the three external datasets

Each `SAM3_INFER_ROOT_<n>` should contain `*_sam3_timeseries.xlsx` files
(produced by running your SAM 3 notebook on the dataset_n videos). Inference
produces window-level probabilities, patient-level scores and predictions.

By default we use Tier 2 for the primary results, but you can swap to Tier 1
or Tier 3 by changing `INFER_TIER`.

In [ ]:
INFER_TIER = 2
BUNDLE_DIR = RUNS_ROOT / f'sam3_tier{INFER_TIER}'

for n, root in [(1, SAM3_INFER_ROOT_1),
                (2, SAM3_INFER_ROOT_2),
                (3, SAM3_INFER_ROOT_3)]:
    if not root.exists():
        print(f'  dataset_{n}: {root} does not exist yet, skipping')
        continue
    out_dir = BUNDLE_DIR / f'infer_dataset_{n}'
    print(f'\n=== Inference dataset_{n} -> {out_dir} ===')
    cmd = [
        sys.executable, str(PIPELINE_DIR / 'inference_sam3.py'),
        '--infer_root', str(root),
        '--bundle_dir', str(BUNDLE_DIR),
        '--out_dir',    str(out_dir),
        '--cody2_root', str(CODY2_REPO),
        '--save_windows',
    ]
    res = subprocess.run(cmd)
    if res.returncode != 0:
        print(f'   [WARN] inference failed for dataset_{n}')

## E. (Optional) Per-site calibration

For the cody-2 paper protocol, calibrate the aggregation rules and thresholds on
a small fixed subset of each external cohort using the cody-2 `tune_threshold_cv.py`
script. The same approach applies here: just point it at the window predictions
from section D.

This section is a placeholder; the calibration code is in cody-2's repo and
doesn't need to be duplicated here.

## F. Reporting

Read the patient-level predictions and produce summary tables for each dataset
and each tier. Compare against ground truth if available (using
`dataset_inference.xlsx` from the cody-2 repo).

In [ ]:
import pandas as pd
frames = []
for tier in TIERS_TO_TRAIN:
    for n in (1, 2, 3):
        p = RUNS_ROOT / f'sam3_tier{tier}' / f'infer_dataset_{n}' / 'reports' / 'tables' / 'inference_patient_predictions.csv'
        if not p.exists():
            continue
        df = pd.read_csv(p)
        df.insert(0, 'tier', tier)
        df.insert(1, 'dataset', f'dataset_{n}')
        frames.append(df)
if frames:
    all_pred = pd.concat(frames, ignore_index=True)
    print(f'Total predictions: {len(all_pred)} rows across tiers and datasets.')
    print(all_pred.groupby(['tier', 'dataset']).size())
else:
    print('No predictions yet. Run section D first.')

---

**Next steps**:

- For deep clinical/feature analysis: open `cody_sam3_features_analysis.ipynb`.
- For per-site calibration: adapt `calibrate_dataset[1,2,3].py` from cody-2.